In [1]:
!pip install -q rasterio pywavelets torchmetrics

In [2]:
import os, sys

open("/kaggle/working/dataset.py","w").write("""import os, warnings, numpy as np, torch
from torch.utils.data import Dataset, DataLoader
import rasterio
warnings.filterwarnings("ignore")

def load_rgb(path):
    with rasterio.open(path) as s:
        r = s.read(3).astype("float32")
        g = s.read(2).astype("float32")
        b = s.read(1).astype("float32")
    rgb = np.stack([r,g,b],-1)
    for c in range(3):
        lo = np.percentile(rgb[...,c], 2)
        hi = np.percentile(rgb[...,c], 98)
        rgb[...,c] = np.clip((rgb[...,c]-lo)/(hi-lo+1e-6), 0, 1)
    return rgb.astype("float32")

def _tifs(d):
    return sorted([f for f in os.listdir(d) if f.lower().endswith((".tif",".tiff"))])

def find_pairs(hr_dir, lr_dir):
    hr = _tifs(hr_dir); lr = _tifs(lr_dir)
    lr_set = set(lr)
    pairs = [(os.path.join(hr_dir,f), os.path.join(lr_dir,f)) for f in hr if f in lr_set]
    if pairs:
        print(f"Exact match: {len(pairs)} pairs"); return pairs
    def pid(name):
        stem = os.path.splitext(name)[0]
        parts = [p for p in stem.split("_") if not (len(p)<=4 and p[:-1].isdigit() and p.endswith('m'))]
        return "_".join(parts)
    lr_map = {pid(f): f for f in lr}
    pairs = [(os.path.join(hr_dir,f), os.path.join(lr_dir,lr_map[pid(f)])) for f in hr if pid(f) in lr_map]
    if pairs:
        print(f"Patch-ID match: {len(pairs)} pairs"); return pairs
    n = min(len(hr),len(lr))
    pairs = [(os.path.join(hr_dir,hr[i]),os.path.join(lr_dir,lr[i])) for i in range(n)]
    print(f"Positional match: {n} pairs"); return pairs

class SRDataset(Dataset):
    def __init__(self, hr_dir, lr_dir, split="train", ratio=0.85, aug=True, max_pairs=20000):
        self.aug = aug and split=="train"
        all_p = find_pairs(hr_dir, lr_dir)
        assert all_p, "No pairs found!"
        if max_pairs and len(all_p) > max_pairs:
            np.random.seed(99)
            idx2 = np.random.choice(len(all_p), max_pairs, replace=False)
            all_p = [all_p[i] for i in sorted(idx2)]
            print(f"Subsampled to {max_pairs} pairs")
        np.random.seed(42)
        idx = np.random.permutation(len(all_p))
        cut = int(len(all_p)*ratio)
        sel = idx[:cut] if split=="train" else idx[cut:]
        self.pairs = [all_p[i] for i in sel]
        print(f"[{split:5s}] {len(self.pairs)} pairs")
    def __len__(self): return len(self.pairs)
    def __getitem__(self, i):
        hp, lp = self.pairs[i]
        hr = torch.from_numpy(load_rgb(hp)).permute(2,0,1)
        lr = torch.from_numpy(load_rgb(lp)).permute(2,0,1)
        if self.aug:
            if torch.rand(1)>.5: hr,lr=torch.flip(hr,[2]),torch.flip(lr,[2])
            if torch.rand(1)>.5: hr,lr=torch.flip(hr,[1]),torch.flip(lr,[1])
            if torch.rand(1)>.5: hr,lr=torch.rot90(hr,1,[1,2]),torch.rot90(lr,1,[1,2])
        return {"lr":lr.clamp(0,1),"hr":hr.clamp(0,1)}

def get_dataloaders(data_dir, batch_size=8, num_workers=2, ratio=0.85, max_pairs=20000):
    hr_dir = os.path.join(data_dir,"HR")
    lr_dir = os.path.join(data_dir,"LR")
    assert os.path.isdir(hr_dir), f"HR not found: {hr_dir}"
    assert os.path.isdir(lr_dir), f"LR not found: {lr_dir}"
    tr = SRDataset(hr_dir, lr_dir, "train", ratio, aug=True,  max_pairs=max_pairs)
    va = SRDataset(hr_dir, lr_dir, "val",   ratio, aug=False, max_pairs=max_pairs)
    s  = tr[0]; lh,lw=s["lr"].shape[1:]; hh,hw=s["hr"].shape[1:]
    scale = hh//lh
    print(f"Scale: x{scale}  (LR={lh}x{lw} -> HR={hh}x{hw})")
    kw = dict(num_workers=num_workers, pin_memory=True, persistent_workers=(num_workers>0))
    return (DataLoader(tr,batch_size=batch_size,shuffle=True,drop_last=True,**kw),
            DataLoader(va,batch_size=1,shuffle=False,**kw), scale)
""")

open("/kaggle/working/vit_isrgan.py","w").write("""import torch, torch.nn as nn, torch.nn.functional as F

class LWB(nn.Module):
    def __init__(self, C):
        super().__init__()
        h = C//2
        self.branch = nn.Sequential(
            nn.Conv2d(h,h,1,bias=False), nn.ReLU(inplace=True),
            nn.Conv2d(h,h,3,1,1,groups=h,bias=False),
            nn.Conv2d(h,h,1,bias=False), nn.ReLU(inplace=True))
    def _shuf(self, x):
        B,C,H,W=x.shape
        return x.view(B,2,C//2,H,W).transpose(1,2).contiguous().view(B,C,H,W)
    def forward(self, x):
        a,b = x.chunk(2,dim=1)
        return self._shuf(torch.cat([a,self.branch(b)],1))

class SEBlock(nn.Module):
    def __init__(self, C, r=16):
        super().__init__()
        r = max(1,C//r)
        self.sq = nn.AdaptiveAvgPool2d(1)
        self.ex = nn.Sequential(nn.Linear(C,r),nn.ReLU(inplace=True),nn.Linear(r,C),nn.Sigmoid())
    def forward(self, x):
        B,C,_,_=x.shape
        return x*self.ex(self.sq(x).view(B,C)).view(B,C,1,1)

class ResBlock(nn.Module):
    def __init__(self, C):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(C,C,3,1,1,bias=False), nn.PReLU(), nn.Conv2d(C,C,3,1,1,bias=False))
    def forward(self, x): return x+self.body(x)

class SSRAM(nn.Module):
    def __init__(self, C):
        super().__init__()
        self.sp = nn.Sequential(
            nn.Conv2d(C,C,3,1,1,bias=False), nn.Conv2d(C,C,5,1,2,bias=False),
            nn.Conv2d(C,C,7,1,3,bias=False), nn.Conv2d(C,C,1,bias=False), nn.Sigmoid())
        self.sc = nn.Sequential(
            nn.Conv2d(C,C,1,bias=False), nn.Conv2d(C,C,(3,1),1,(1,0),bias=False),
            nn.Conv2d(C,C,(5,1),1,(2,0),bias=False), nn.Conv2d(C,C,1,bias=False), nn.Sigmoid())
        self.ffn = nn.Sequential(
            nn.Conv2d(C,C,1,bias=False), nn.LeakyReLU(0.2,inplace=True),
            nn.Conv2d(C,C,1,bias=False), nn.InstanceNorm2d(C))
        self.out = nn.Conv2d(C,C,1,bias=False)
        self.a = nn.Parameter(torch.ones(1))
        self.b = nn.Parameter(torch.ones(1))
    def forward(self, x):
        f = self.a*self.sp(x)*x + self.b*self.sc(x)*x
        return x+self.out(self.ffn(f))

class ViTBlock(nn.Module):
    def __init__(self, d, h=8, mlp=4, drop=0.0):
        super().__init__()
        self.n1=nn.LayerNorm(d); self.n2=nn.LayerNorm(d)
        self.at=nn.MultiheadAttention(d,h,dropout=drop,batch_first=True)
        self.ff=nn.Sequential(nn.Linear(d,d*mlp),nn.GELU(),nn.Dropout(drop),
                              nn.Linear(d*mlp,d),nn.Dropout(drop))
    def forward(self, x):
        h,_=self.at(self.n1(x),self.n1(x),self.n1(x)); x=x+h
        return x+self.ff(self.n2(x))

class ViTModule(nn.Module):
    def __init__(self, C, h=8, depth=8):
        super().__init__()
        self.blocks=nn.ModuleList([ViTBlock(C,h) for _ in range(depth)])
    def _pos(self, N, C, dev):
        p=torch.arange(N,device=dev).unsqueeze(1).float()
        i=torch.arange(C,device=dev).unsqueeze(0).float()
        a=p/(10000**(2*(i//2)/C))
        a[:,0::2]=a[:,0::2].sin(); a[:,1::2]=a[:,1::2].cos()
        return a.unsqueeze(0)
    def forward(self, x):
        B,C,H,W=x.shape; t=x.flatten(2).transpose(1,2)
        t=t+self._pos(H*W,C,x.device)
        for b in self.blocks: t=b(t)
        return t.transpose(1,2).view(B,C,H,W)+x

class Upsampler(nn.Module):
    def __init__(self, C, s=2):
        super().__init__()
        self.c=nn.Conv2d(C,C*s*s,3,1,1); self.p=nn.PixelShuffle(s); self.a=nn.PReLU()
    def forward(self,x): return self.a(self.p(self.c(x)))

class Generator(nn.Module):
    def __init__(self, C=64, scale_factor=2):
        super().__init__()
        self.head=nn.Sequential(nn.Conv2d(3,C,3,1,1),nn.PReLU())
        self.lwb1=LWB(C); self.se1=SEBlock(C)
        self.lwb2=LWB(C); self.se2=SEBlock(C)
        self.res1=ResBlock(C); self.se3=SEBlock(C)
        self.res2=ResBlock(C); self.se4=SEBlock(C)
        self.res3=ResBlock(C)
        self.ssram=SSRAM(C)
        self.mid=nn.Sequential(nn.Conv2d(C,C,3,1,1),nn.PReLU())
        self.vit=ViTModule(C,h=8,depth=8)
        self.vp=nn.Conv2d(C,C,3,1,1)
        n_ups = 1 if scale_factor==2 else 2
        self.upsamplers=nn.Sequential(*[Upsampler(C,2) for _ in range(n_ups)])
        self.tail=nn.Conv2d(C,3,3,1,1)
    def forward(self, x):
        f=self.head(x)
        lw=self.se2(self.lwb2(self.se1(self.lwb1(f))))
        r=self.se3(self.res1(lw)); r=self.se4(self.res2(r)); r=self.res3(r)
        r=self.mid(self.ssram(r))
        vf=self.vp(self.vit(r))+lw
        return torch.sigmoid(self.tail(self.upsamplers(vf)))

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        def b(ic,oc,s=1):
            return nn.Sequential(nn.Conv2d(ic,oc,3,s,1,bias=False),
                                 nn.BatchNorm2d(oc),nn.LeakyReLU(0.2,True))
        self.net=nn.Sequential(
            nn.Conv2d(3,64,3,1,1),nn.LeakyReLU(0.2,True),
            b(64,64,2),b(64,128,1),b(128,128,2),b(128,256,1),
            b(256,256,2),b(256,512,1),b(512,512,2))
        self.clf=nn.Sequential(
            nn.AdaptiveAvgPool2d(1),nn.Flatten(),
            nn.Linear(512,1024),nn.LeakyReLU(0.2,True),
            nn.Linear(1024,1),nn.Sigmoid())
    def forward(self,x): return self.clf(self.net(x))

def build_model(scale_factor=2):
    G=Generator(C=64, scale_factor=scale_factor)
    D=Discriminator()
    print(f"Generator: {sum(p.numel() for p in G.parameters())/1e6:.2f}M params  |  scale=x{scale_factor}")
    return G,D
""")

open("/kaggle/working/wavelet_utils.py","w").write("""import torch, torch.nn.functional as F
_HAAR=torch.tensor([[[ 0.5, 0.5],[ 0.5, 0.5]],[[ 0.5, 0.5],[-0.5,-0.5]],[[ 0.5,-0.5],[ 0.5,-0.5]],[[ 0.5,-0.5],[-0.5, 0.5]]])
def dwt(x):
    B,C,H,W=x.shape; f=_HAAR.unsqueeze(1).to(x.device)
    o=F.conv2d(x.view(B*C,1,H,W),f,stride=2).view(B,C,4,H//2,W//2)
    return o[:,:,0],o[:,:,1],o[:,:,2],o[:,:,3]
def idwt(LL,LH,HL,HH):
    B,C,H,W=LL.shape; f=_HAAR.unsqueeze(1).to(LL.device)
    s=torch.stack([LL,LH,HL,HH],2).view(B*C,4,H,W)
    out=torch.zeros(B*C,1,H*2,W*2,device=LL.device)
    for i in range(4): out=out+F.conv_transpose2d(s[:,i:i+1],f[i:i+1],stride=2)
    return out.view(B,C,H*2,W*2)
def hf_upscale(LH,HL,HH,scale=2):
    kw=dict(scale_factor=scale,mode="bicubic",align_corners=False)
    return F.interpolate(LH,**kw),F.interpolate(HL,**kw),F.interpolate(HH,**kw)
""")

open("/kaggle/working/wavediffur.py","w").write("""import torch, math
from wavelet_utils import dwt, idwt, hf_upscale
class WaveDiffUR:
    def __init__(self, G, device="cuda"):
        self.G=G.to(device).eval(); self.dev=device
    @torch.no_grad()
    def _step(self, lr):
        lr=lr.to(self.dev); LL,LH,HL,HH=dwt(lr)
        LL_sr=self.G(LL); LH_sr,HL_sr,HH_sr=hf_upscale(LH,HL,HH,scale=2)
        return idwt(LL_sr,LH_sr,HL_sr,HH_sr).clamp(0,1)
    @torch.no_grad()
    def run(self, lr_image, target_scale=4):
        assert target_scale in [4,16,64]
        n=int(math.log(target_scale,4)); cur=lr_image.to(self.dev)
        print(f"WaveDiffUR: {list(cur.shape[-2:])} -> x{target_scale}")
        for i in range(n):
            cur=self._step(cur)
            print(f"  Step {i+1}: x{4**(i+1)} shape={list(cur.shape[-2:])}")
        return cur
""")

open("/kaggle/working/losses.py","w").write("""import torch, torch.nn as nn, torch.nn.functional as F
class Losses(nn.Module):
    def __init__(self, lp=0.01, lperc=0.1, ladv=0.001):
        super().__init__()
        self.lp=lp; self.lperc=lperc; self.ladv=ladv
        import torchvision.models as M
        vgg=M.vgg19(pretrained=True).features[:18].eval()
        for p in vgg.parameters(): p.requires_grad=False
        self.vgg=vgg; self.mse=nn.MSELoss()
    def _vgg_feats(self, x):
        with torch.no_grad(): return self.vgg(x)
    def G_loss(self, sr, hr, fake_score):
        pix=self.mse(sr,hr)
        perc=F.mse_loss(self._vgg_feats(sr),self._vgg_feats(hr))
        adv=-torch.mean(torch.log(fake_score+1e-8))
        tot=self.lp*pix+self.lperc*perc+self.ladv*adv
        return tot,{"pix":pix.item(),"perc":perc.item(),"adv":adv.item()}
    def D_loss(self,real,fake):
        return -torch.mean(torch.log(real+1e-8))-torch.mean(torch.log(1-fake+1e-8))
""")

open("/kaggle/working/train.py","w").write("""import os,json,shutil,numpy as np,torch
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler,autocast
from torchmetrics.image import PeakSignalNoiseRatio as PSNR,StructuralSimilarityIndexMeasure as SSIM
from dataset import get_dataloaders
from vit_isrgan import build_model
from losses import Losses

def _save(ck,out_dir,tag):
    p=f"{out_dir}/{tag}.pth"; torch.save(ck,p)
    shutil.copy(p,f"/kaggle/working/{tag}_backup.pth"); return p

def train(data_dir,out_dir="/kaggle/working/checkpoints",epochs=10,batch_size=1,
          lr=5e-5,device="cuda",resume=None,save_every_n_batches=100,max_pairs=20000):
    os.makedirs(out_dir,exist_ok=True)
    dev=torch.device(device if torch.cuda.is_available() else "cpu")
    print(f"Device: {dev}")
    trl,val,scale=get_dataloaders(data_dir,batch_size=batch_size,
                                   num_workers=2,max_pairs=max_pairs)
    print(f"Detected scale: x{scale}")
    G,D=build_model(scale_factor=scale); G,D=G.to(dev),D.to(dev)
    crit=Losses().to(dev)
    oG=Adam(G.parameters(),lr=lr,betas=(0.9,0.999))
    oD=Adam(D.parameters(),lr=lr,betas=(0.9,0.999))
    sG=CosineAnnealingLR(oG,T_max=100,eta_min=1e-6)
    sD=CosineAnnealingLR(oD,T_max=100,eta_min=1e-6)
    scaler=GradScaler()
    psnr_f=PSNR(data_range=1.0).to(dev)
    ssim_f=SSIM(data_range=1.0).to(dev)
    start=0; best=0.0
    rp=resume or "/kaggle/working/last_backup.pth"
    if os.path.exists(rp):
        print(f"Resuming from {rp}")
        ck=torch.load(rp,map_location=dev,weights_only=False)
        G.load_state_dict(ck["G"]); D.load_state_dict(ck["D"])
        oG.load_state_dict(ck["oG"]); oD.load_state_dict(ck["oD"])
        start=ck["epoch"]+1; best=ck.get("psnr",0.0)
        saved_scale=ck.get("scale",scale)
        if saved_scale!=scale:
            scale=saved_scale; G,D=build_model(scale_factor=scale)
            G,D=G.to(dev),D.to(dev)
            G.load_state_dict(ck["G"]); D.load_state_dict(ck["D"])
            oG.load_state_dict(ck["oG"]); oD.load_state_dict(ck["oD"])
        print(f"  Epoch {start}, best PSNR so far={best:.3f}")
    else:
        print("Starting fresh.")
    log=[]
    for ep in range(start,epochs):
        G.train(); D.train(); gL=dL=0.0
        for bi,b in enumerate(trl):
            li=b["lr"].to(dev); hi=b["hr"].to(dev)
            oD.zero_grad()
            with autocast(): ld=crit.D_loss(D(hi),D(G(li).detach()))
            scaler.scale(ld).backward(); scaler.step(oD)
            oG.zero_grad()
            with autocast(): sr=G(li); lg,comp=crit.G_loss(sr,hi,D(sr))
            scaler.scale(lg).backward(); scaler.step(oG); scaler.update()
            gL+=lg.item(); dL+=ld.item()
            if bi%50==0:
                print(f"Ep{ep}[{bi}/{len(trl)}] G={lg.item():.4f} "
                      f"pix={comp['pix']:.4f} perc={comp['perc']:.4f} D={ld.item():.4f}")
            if save_every_n_batches and (bi+1)%save_every_n_batches==0:
                _save({"epoch":ep,"scale":scale,"G":G.state_dict(),"D":D.state_dict(),
                       "oG":oG.state_dict(),"oD":oD.state_dict(),"psnr":best},out_dir,"last")
                print(f"  [saved @batch {bi}]")
        sG.step(); sD.step()
        G.eval(); vp=[]; vs=[]
        with torch.no_grad():
            for b in val:
                sr=G(b["lr"].to(dev)).clamp(0,1); hr=b["hr"].to(dev)
                vp.append(psnr_f(sr,hr).item()); vs.append(ssim_f(sr,hr).item())
        torch.cuda.empty_cache()
        pm=np.mean(vp); sm=np.mean(vs)
        print(f"\\n>>> Ep{ep}: PSNR={pm:.3f}  SSIM={sm:.4f}\\n")
        log.append({"epoch":ep,"psnr":pm,"ssim":sm,
                    "lG":gL/len(trl),"lD":dL/len(trl)})
        ck={"epoch":ep,"scale":scale,"G":G.state_dict(),"D":D.state_dict(),
            "oG":oG.state_dict(),"oD":oD.state_dict(),"psnr":pm}
        _save(ck,out_dir,"last")
        print("  Saved - SAFE TO SAVE VERSION NOW")
        if pm>best:
            best=pm; _save(ck,out_dir,"best")
            print(f"  New best PSNR={best:.3f}")
        with open(f"{out_dir}/log.json","w") as f:
            json.dump(log,f,indent=2)
    print(f"Done. Best PSNR={best:.3f}"); return G,log
""")

open("/kaggle/working/evaluate.py","w").write("""import numpy as np, torch, pywt
from torchmetrics.image import PeakSignalNoiseRatio as PSNR
from torchmetrics.image import StructuralSimilarityIndexMeasure as SSIM
def _to_3d(x):
    x=np.asarray(x,dtype=np.float32)
    return x[...,None] if x.ndim==2 else x
def sam(s,h):
    s=_to_3d(s); h=_to_3d(h)
    dot=np.sum(s*h,axis=-1)
    denom=np.linalg.norm(s,axis=-1)*np.linalg.norm(h,axis=-1)+1e-8
    return float(np.mean(np.arccos(np.clip(dot/denom,-1,1))))
def scc(s,h):
    s=_to_3d(s).reshape(-1,3); h=_to_3d(h).reshape(-1,3)
    s=s-s.mean(0); h=h-h.mean(0)
    num=np.sum(s*h,axis=0)
    den=np.sqrt(np.sum(s*s,0)*np.sum(h*h,0))+1e-8
    return float(np.mean(num/den))
def sre(s,h): return float(np.mean((_to_3d(s)-_to_3d(h))**2))
def niqe(img):
    img=_to_3d(img)
    g=0.299*img[...,0]+0.587*img[...,1]+0.114*img[...,2]
    _,(ch,cv,cd)=pywt.dwt2(g,"haar")
    return float(1.0/(np.std(ch)+np.std(cv)+np.std(cd)+1e-6))
def ag(img):
    img=_to_3d(img)
    g=0.299*img[...,0]+0.587*img[...,1]+0.114*img[...,2]
    gx=np.diff(g,axis=1); gy=np.diff(g,axis=0)
    return float(np.mean(np.sqrt(gx[:-1]**2+gy[:,:-1]**2+1e-8)))
def evaluate(G,val_loader,device="cuda"):
    dev=torch.device(device if torch.cuda.is_available() else "cpu")
    G=G.to(dev).eval()
    psnr_f=PSNR(data_range=1.0).to(dev)
    ssim_f=SSIM(data_range=1.0).to(dev)
    M={k:[] for k in ["psnr","ssim","sam","scc","sre","niqe","ag"]}
    with torch.no_grad():
        for b in val_loader:
            lr=b["lr"].to(dev); hr=b["hr"].to(dev)
            sr=G(lr).clamp(0,1)
            M["psnr"].append(psnr_f(sr,hr).item())
            M["ssim"].append(ssim_f(sr,hr).item())
            sn=sr[0].permute(1,2,0).cpu().numpy()
            hn=hr[0].permute(1,2,0).cpu().numpy()
            M["sam"].append(sam(sn,hn)); M["scc"].append(scc(sn,hn))
            M["sre"].append(sre(sn,hn)); M["niqe"].append(niqe(sn))
            M["ag"].append(ag(sn))
    print("\\n==== Evaluation Results ====")
    R={}
    for k,v in M.items():
        R[k]=float(np.mean(v))
        up=k in ["psnr","ssim","scc","ag"]
        print(f"  {k.upper():6s} {'up' if up else 'dn'} : {R[k]:.4f}")
    print("============================")
    return R
""")

open("/kaggle/working/infer.py","w").write("""import torch, numpy as np, os
from PIL import Image
from dataset import load_rgb
from vit_isrgan import build_model
from wavediffur import WaveDiffUR
def infer(tif_path,out_path,ckpt,target_scale=4,device="cuda"):
    os.makedirs(os.path.dirname(out_path) or ".",exist_ok=True)
    dev=torch.device(device if torch.cuda.is_available() else "cpu")
    ck=torch.load(ckpt,map_location=dev,weights_only=False)
    G,_=build_model(scale_factor=ck.get("scale",2))
    G.load_state_dict(ck["G"]); G.eval().to(dev)
    pipe=WaveDiffUR(G,str(dev))
    rgb=load_rgb(tif_path)
    lr=torch.from_numpy(rgb).permute(2,0,1).unsqueeze(0)
    sr=pipe.run(lr,target_scale)
    out=(sr.squeeze(0).permute(1,2,0).cpu().numpy()*255).clip(0,255).astype(np.uint8)
    Image.fromarray(out).save(out_path)
    print(f"Saved {out_path}  ({out.shape[0]}x{out.shape[1]})")
    return out
""")

sys.path.insert(0, "/kaggle/working")
print("Files written:")
for fn in ["dataset","vit_isrgan","wavelet_utils","wavediffur","losses","train","evaluate","infer"]:
    path = f"/kaggle/working/{fn}.py"
    size = os.path.getsize(path) if os.path.exists(path) else 0
    print(f"  {'OK' if size>100 else 'FAIL':4s} {fn}.py ({size} bytes)")

Files written:
  OK   dataset.py (3676 bytes)
  OK   vit_isrgan.py (5478 bytes)
  OK   wavelet_utils.py (823 bytes)
  OK   wavediffur.py (803 bytes)
  OK   losses.py (923 bytes)
  OK   train.py (4212 bytes)
  OK   evaluate.py (2203 bytes)
  OK   infer.py (862 bytes)


In [3]:
# Find the checkpoint from Version 4 output
import os, torch

dataset_dir = "/kaggle/input/datasets/shomailezahra/epochs-48"   # ← your new dataset name
ckpt_path = None
for f in sorted(os.listdir(dataset_dir)):
    if f.endswith(".pth"):
        ckpt_path = os.path.join(dataset_dir, f)
        print(f"Found: {f}")
        break

print(f"Checkpoint: {ckpt_path}")
ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
print(f"Epoch: {ck.get('epoch','?')}")
print(f"PSNR:  {ck.get('psnr','?')}")

# Verify the key names look right
sample_keys = [k for k in ck["G"].keys() if "ups" in k]
print(f"Upsampler keys now: {sample_keys}")

Found: last_backup (6).pth
Checkpoint: /kaggle/input/datasets/shomailezahra/epochs-48/last_backup (6).pth
Epoch: 47
PSNR:  23.754938597043356
Upsampler keys now: ['upsamplers.0.c.weight', 'upsamplers.0.c.bias', 'upsamplers.0.a.weight']


In [4]:
from train import train

G, log = train(
    data_dir             = "/kaggle/input/datasets/ibtehajali1/environmental-data-tiff-big3",
    out_dir              = "/kaggle/working/checkpoints",
    epochs               = 50,        # ← will run from 45 to 50 = 5 more epochs
    batch_size           = 1,
    lr                   = 5e-5,
    device               = "cuda",
    resume               = ckpt_path, # ← pass explicitly this time
    save_every_n_batches = 100,
    max_pairs            = 20000,
)

Device: cuda
Patch-ID match: 333564 pairs
Subsampled to 20000 pairs
[train] 17000 pairs
Patch-ID match: 333564 pairs
Subsampled to 20000 pairs
[val  ] 3000 pairs
Scale: x2  (LR=64x64 -> HR=128x128)
Detected scale: x2
Generator: 1.25M params  |  scale=x2
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:02<00:00, 198MB/s]


Resuming from /kaggle/input/datasets/shomailezahra/epochs-48/last_backup (6).pth
  Epoch 48, best PSNR so far=23.755
Ep48[0/17000] G=0.1150 pix=0.0047 perc=1.1427 D=1.3805
Ep48[50/17000] G=0.1400 pix=0.0142 perc=1.3916 D=1.3839
  [saved @batch 99]
Ep48[100/17000] G=0.1187 pix=0.0067 perc=1.1795 D=1.4005
Ep48[150/17000] G=0.1206 pix=0.0069 perc=1.1982 D=1.3902
  [saved @batch 199]
Ep48[200/17000] G=0.0894 pix=0.0023 perc=0.8870 D=1.3804
Ep48[250/17000] G=0.1016 pix=0.0031 perc=1.0086 D=1.3883
  [saved @batch 299]
Ep48[300/17000] G=0.1242 pix=0.0072 perc=1.2347 D=1.3786
Ep48[350/17000] G=0.0631 pix=0.0021 perc=0.6243 D=1.3853
  [saved @batch 399]
Ep48[400/17000] G=0.0815 pix=0.0031 perc=0.8082 D=1.3932
Ep48[450/17000] G=0.1018 pix=0.0080 perc=1.0103 D=1.3917
  [saved @batch 499]
Ep48[500/17000] G=0.1778 pix=0.0113 perc=1.7705 D=1.3883
Ep48[550/17000] G=0.0789 pix=0.0023 perc=0.7820 D=1.3931
  [saved @batch 599]
Ep48[600/17000] G=0.1029 pix=0.0039 perc=1.0218 D=1.3858
Ep48[650/17000] G=0.

In [5]:
import sys, os, torch, numpy as np, pywt
sys.path.insert(0, "/kaggle/working")

for mod in list(sys.modules.keys()):
    if any(x in mod for x in ["evaluate","vit_is","dataset"]):
        del sys.modules[mod]

from vit_isrgan import build_model
from dataset import get_dataloaders
from torchmetrics.image import PeakSignalNoiseRatio as PSNR
from torchmetrics.image import StructuralSimilarityIndexMeasure as SSIM

# Find best checkpoint
ckpt_path = None
for path in ["/kaggle/working/checkpoints/best.pth",
             "/kaggle/working/checkpoints/last.pth",
             "/kaggle/working/best_backup.pth",
             "/kaggle/working/last_backup.pth"]:
    if os.path.exists(path):
        ckpt_path = path
        print(f"Using: {path}")
        break

assert ckpt_path, "No checkpoint found"
ck = torch.load(ckpt_path, map_location="cuda", weights_only=False)
scale = ck.get("scale", 2)
epoch = ck.get("epoch", "?")
psnr_saved = ck.get("psnr", 0)
print(f"Epoch={epoch}  PSNR={psnr_saved:.3f}  scale=x{scale}")

G, _ = build_model(scale_factor=scale)
G.load_state_dict(ck["G"])
G = G.cuda().eval()

_, val_loader, _ = get_dataloaders(
    "/kaggle/input/datasets/ibtehajali1/environmental-data-tiff-big3",
    batch_size=1, num_workers=2, max_pairs=20000)

psnr_f = PSNR(data_range=1.0).cuda()
ssim_f = SSIM(data_range=1.0).cuda()

def _to_3d(x):
    x = np.asarray(x, dtype=np.float32)
    return x[..., None] if x.ndim == 2 else x

def sam(s, h):
    s=_to_3d(s); h=_to_3d(h)
    dot = np.sum(s*h, axis=-1)
    denom = np.linalg.norm(s,axis=-1)*np.linalg.norm(h,axis=-1)+1e-8
    return float(np.mean(np.arccos(np.clip(dot/denom,-1,1))))

def scc(s, h):
    s=_to_3d(s).reshape(-1,3); h=_to_3d(h).reshape(-1,3)
    s=s-s.mean(0); h=h-h.mean(0)
    num=np.sum(s*h,axis=0)
    den=np.sqrt(np.sum(s*s,0)*np.sum(h*h,0))+1e-8
    return float(np.mean(num/den))

def sre(s, h): return float(np.mean((_to_3d(s)-_to_3d(h))**2))

def niqe(img):
    img=_to_3d(img)
    g=0.299*img[...,0]+0.587*img[...,1]+0.114*img[...,2]
    _,(ch,cv,cd)=pywt.dwt2(g,"haar")
    return float(1.0/(np.std(ch)+np.std(cv)+np.std(cd)+1e-6))

def ag(img):
    img=_to_3d(img)
    g=0.299*img[...,0]+0.587*img[...,1]+0.114*img[...,2]
    gx=np.diff(g,axis=1); gy=np.diff(g,axis=0)
    return float(np.mean(np.sqrt(gx[:-1]**2+gy[:,:-1]**2+1e-8)))

M = {k:[] for k in ["psnr","ssim","sam","scc","sre","niqe","ag"]}
print("Evaluating...")
with torch.no_grad():
    for i, b in enumerate(val_loader):
        lr=b["lr"].cuda(); hr=b["hr"].cuda()
        sr=G(lr).clamp(0,1)
        M["psnr"].append(psnr_f(sr,hr).item())
        M["ssim"].append(ssim_f(sr,hr).item())
        sn=sr[0].permute(1,2,0).cpu().numpy()
        hn=hr[0].permute(1,2,0).cpu().numpy()
        M["sam"].append(sam(sn,hn)); M["scc"].append(scc(sn,hn))
        M["sre"].append(sre(sn,hn)); M["niqe"].append(niqe(sn))
        M["ag"].append(ag(sn))
        if (i+1) % 500 == 0:
            print(f"  {i+1}/{len(val_loader)} done...")

results = {k: float(np.mean(v)) for k,v in M.items()}

print(f"\n{'='*45}")
print(f"  FINAL RESULTS — Epoch {epoch}")
print(f"{'='*45}")
for k, v in results.items():
    up = k in ["psnr","ssim","scc","ag"]
    print(f"  {k.upper():6s} {'↑' if up else '↓'} : {v:.4f}")
print(f"{'='*45}")

# Save results so later cells can use them even if kernel restarts
import json
os.makedirs("/kaggle/working/results", exist_ok=True)
with open("/kaggle/working/results/metrics.json","w") as f:
    json.dump({"results": results, "epoch": epoch, "scale": scale}, f)
print("Metrics saved to /kaggle/working/results/metrics.json")

Using: /kaggle/working/checkpoints/best.pth
Epoch=49  PSNR=23.755  scale=x2
Generator: 1.25M params  |  scale=x2
Patch-ID match: 333564 pairs
Subsampled to 20000 pairs
[train] 17000 pairs
Patch-ID match: 333564 pairs
Subsampled to 20000 pairs
[val  ] 3000 pairs
Scale: x2  (LR=64x64 -> HR=128x128)
Evaluating...
  500/3000 done...
  1000/3000 done...
  1500/3000 done...
  2000/3000 done...
  2500/3000 done...
  3000/3000 done...

  FINAL RESULTS — Epoch 49
  PSNR   ↑ : 23.7554
  SSIM   ↑ : 0.7672
  SAM    ↓ : 0.1381
  SCC    ↑ : 0.9485
  SRE    ↓ : 0.0063
  NIQE   ↓ : 9.6354
  AG     ↑ : 0.0519
Metrics saved to /kaggle/working/results/metrics.json


In [6]:
import sys, os, torch, numpy as np, json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image

sys.path.insert(0, "/kaggle/working")
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ["wavediffur","wavelet","vit_is","dataset"]):
        del sys.modules[mod]
sys.path.insert(0, "/kaggle/working")

from dataset import load_rgb
from vit_isrgan import build_model
from wavediffur import WaveDiffUR

os.makedirs("/kaggle/working/results", exist_ok=True)

with open("/kaggle/working/results/metrics.json") as f:
    saved = json.load(f)
results = saved["results"]
epoch   = saved["epoch"]
scale   = saved["scale"]

ckpt_path = None
for path in ["/kaggle/working/checkpoints/best.pth",
             "/kaggle/working/checkpoints/last.pth",
             "/kaggle/working/best_backup.pth",
             "/kaggle/working/last_backup.pth"]:
    if os.path.exists(path):
        ckpt_path = path; break

ck = torch.load(ckpt_path, map_location="cuda", weights_only=False)
G, _ = build_model(scale_factor=scale)
G.load_state_dict(ck["G"]); G.eval().cuda()
pipe = WaveDiffUR(G, "cuda")

lr_dir = "/kaggle/input/datasets/ibtehajali1/environmental-data-tiff-big3/LR"
hr_dir = "/kaggle/input/datasets/ibtehajali1/environmental-data-tiff-big3/HR"
lr_files = sorted(os.listdir(lr_dir))
hr_files = sorted(os.listdir(hr_dir))
selected = [0, 100, 500]

print("Generating SR images...")
for idx in selected:
    lr_rgb = load_rgb(os.path.join(lr_dir, lr_files[idx]))
    hr_rgb = load_rgb(os.path.join(hr_dir, hr_files[idx]))
    lr_t   = torch.from_numpy(lr_rgb).permute(2,0,1).unsqueeze(0)

    with torch.no_grad():
        sr_vit = G(lr_t.cuda()).clamp(0,1)
        sr_np  = sr_vit[0].permute(1,2,0).cpu().numpy()

    sr_wave = pipe.run(lr_t, target_scale=4)
    sw_np   = sr_wave[0].permute(1,2,0).cpu().numpy()

    Image.fromarray((lr_rgb*255).clip(0,255).astype(np.uint8)).save(
        f"/kaggle/working/results/lr_{idx}.png")
    Image.fromarray((sr_np*255).clip(0,255).astype(np.uint8)).save(
        f"/kaggle/working/results/vit_{idx}.png")
    Image.fromarray((sw_np*255).clip(0,255).astype(np.uint8)).save(
        f"/kaggle/working/results/wave_{idx}.png")
    Image.fromarray((hr_rgb*255).clip(0,255).astype(np.uint8)).save(
        f"/kaggle/working/results/hr_{idx}.png")
    print(f"  Patch {idx} done")

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
col_titles = [
    "LR Input\n64x64 (10m/px)",
    "ViT-ISRGAN x2 SR\n128x128 (5m/px)",
    "WaveDiffUR x4\n256x256 (2.5m/px)",
    "HR Ground Truth\n128x128 (5m/px)"
]
for col, title in enumerate(col_titles):
    axes[0][col].set_title(title, fontsize=9, fontweight="bold")

for row, idx in enumerate(selected):
    imgs = [
        np.array(Image.open(f"/kaggle/working/results/lr_{idx}.png")).astype(np.float32)/255,
        np.array(Image.open(f"/kaggle/working/results/vit_{idx}.png")).astype(np.float32)/255,
        np.array(Image.open(f"/kaggle/working/results/wave_{idx}.png")).astype(np.float32)/255,
        np.array(Image.open(f"/kaggle/working/results/hr_{idx}.png")).astype(np.float32)/255,
    ]
    for col, img in enumerate(imgs):
        axes[row][col].imshow(img.clip(0,1))
        axes[row][col].axis("off")
    axes[row][0].set_ylabel(f"Sample {row+1}", fontsize=10, fontweight="bold")

plt.suptitle(
    f"WaveDiffUR + ViT-ISRGAN | Epoch {epoch} | "
    f"PSNR={results['psnr']:.3f} dB | SSIM={results['ssim']:.4f}\n"
    f"Task: x2 SR (64->128). WaveDiffUR cascades to x4 (256).",
    fontsize=9, y=1.01)
plt.tight_layout()
plt.savefig("/kaggle/working/results/visual_comparison.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: visual_comparison.png")

Generator: 1.25M params  |  scale=x2
Generating SR images...
WaveDiffUR: [64, 64] -> x4
  Step 1: x4 shape=[128, 128]
  Patch 0 done
WaveDiffUR: [64, 64] -> x4
  Step 1: x4 shape=[128, 128]
  Patch 100 done
WaveDiffUR: [64, 64] -> x4
  Step 1: x4 shape=[128, 128]
  Patch 500 done
Saved: visual_comparison.png


In [7]:
import json, os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

# Load results saved by Cell 4 — no dependency on any variable
with open("/kaggle/working/results/metrics.json") as f:
    saved = json.load(f)
results = saved["results"]
epoch   = saved["epoch"]

paper = {
    "Bicubic":            (35.883, 0.897),
    "VDSR":               (36.685, 0.904),
    "EDSR":               (38.713, 0.930),
    "SRGAN":              (38.876, 0.931),
    "ESRGAN":             (39.009, 0.937),
    "ViT-ISRGAN\n(paper)":(39.091, 0.941),
}
models    = list(paper.keys()) + [f"Ours\n(x2,RGB,{epoch}ep)"]
psnr_vals = [v[0] for v in paper.values()] + [results["psnr"]]
ssim_vals = [v[1] for v in paper.values()] + [results["ssim"]]
colors    = ["#95a5a6"]*6 + ["#e74c3c"]

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, vals, title, ylabel, ylim in [
    (axes[0], psnr_vals, "PSNR (dB) ↑", "PSNR (dB)", (15, 42)),
    (axes[1], ssim_vals, "SSIM ↑",       "SSIM",      (0.4, 1.0)),
]:
    bars = ax.bar(models, vals, color=colors, edgecolor="black", linewidth=0.5)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_ylabel(ylabel)
    ax.set_ylim(ylim)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height()+(0.2 if "PSNR" in ylabel else 0.005),
                f"{val:.3f}", ha="center", va="bottom", fontsize=7)
    ax.tick_params(axis="x", rotation=30)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle(
    "NOTE: Gray bars = paper (x4 SR, 4-band multispectral, 100 epochs)\n"
    "Red bar = ours (x2 SR, RGB only, limited epochs) — different task, not a direct comparison\n"
    "Contribution: ViT-ISRGAN backbone integrated inside WaveDiffUR cascade framework",
    fontsize=8, style="italic", color="darkred")
plt.tight_layout()
plt.savefig("/kaggle/working/results/metrics_bar.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: metrics_bar.png")
print("\nAll done! Files in /kaggle/working/results/:")
for f in sorted(os.listdir("/kaggle/working/results/")):
    print(f"  {f}")

Saved: metrics_bar.png

All done! Files in /kaggle/working/results/:
  hr_0.png
  hr_100.png
  hr_500.png
  lr_0.png
  lr_100.png
  lr_500.png
  metrics.json
  metrics_bar.png
  visual_comparison.png
  vit_0.png
  vit_100.png
  vit_500.png
  wave_0.png
  wave_100.png
  wave_500.png
